# Tome C3, Chapter 4: Undecidable Problems in Mathematics (Python companion)

Exercises first, solutions at the end. No program can decide the problems of this chapter, but programs can explore them: we search for matches of Post's dominoes (and find one that needs $206$ dominoes), turn a Turing machine into a rewriting system and then into dominoes, tile squares with the tiles of Theorem 4.11, watch Conway's PRIMEGAME produce the primes, iterate Collatz-type maps including the six-state Antihydra, and check the Pell-equation lemmas behind Hilbert's tenth problem.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, math, time, random
from collections import deque
from fractions import Fraction
import matplotlib.pyplot as plt
random.seed(4)

## Exercises

**P4.1 (★) Searching for matches.** Write a breadth-first search for matches of a PCP instance, whose states are the *overhang*, the part of one row that sticks out beyond the other. Check the match of Figure 4.1, show that $3,2,3,1$ is the shortest match in Exercise 4.2, and find the shortest match of the instance $(1000,0)$, $(01,0)$, $(1,101)$, $(00,001)$.

**P4.2 (★★) A machine as a rewriting system.** Build the semi-Thue system $R_M$ of Theorem 4.2 for the palindrome machine of Chapter 3, check on all short words that $[q_0w]\Rightarrow^*H$ exactly when $w$ is a palindrome, and check that every word met has at most one successor.

**P4.3 (★★) From derivations to dominoes.** Build the MPCP instance of Theorem 4.6 from $R_M$ and an input, construct the match from the derivation as in the proof, and verify that its two rows agree. Then apply Step 2 of the proof (the stars) and verify again.

**P4.4 (★★) Tiling squares.** Build the tile set $T_M$ of Theorem 4.11 and tile the squares $[-n,n]^2$ with $t_0$ at the origin by backtracking. Compare a machine that never halts with the two-state busy beaver, which halts after $6$ steps.

**P4.5 (★) FRACTRAN.** Run PRIMEGAME and list the powers of $2$ it meets, with the number of steps. Check Exercise 4.5 on small values. Then compile a counter machine into FRACTRAN as in the proof of Theorem 4.16, and run it.

**P4.6 (★) Collatz-type maps.** Turn a FRACTRAN program into a map $n\mapsto a_{n\bmod m}n$ (Corollary 4.17). Then compute record stopping times of the Collatz map, and run the Antihydra iteration of Section 4.6 for $10^5$ steps: does $b$ come close to $-1$?

**P4.7 (★★) Pell sequences.** Compute $x_n(a)$, $y_n(a)$, check Lemma 4.23 by brute force for small $a$, Lemma 4.24 and Exercise 4.8, and recover $k^n$ from the Pell sequences as in the proof of Proposition 4.26.

**P4.8 (★) Diophantine definitions.** Write the semi-decision procedure of Proposition 4.19(c) for an equation, and apply it to $x^2-dy^2=1$ for several $d$; why is $d=61$ so slow? Then apply the trick of Corollary 4.22 to the Diophantine definition of composite numbers, and list the positive values of the resulting polynomial.

## Solutions

### P4.1

In [ ]:
def pcp_search(dominoes, max_states=10**6):
    # Breadth-first search; a state is (which row is ahead, overhang). Returns a shortest match or None.
    seen, queue = {}, deque()
    def step(state, i):
        side, o = state
        t, b = dominoes[i]
        top, bot = (o + t, b) if side == 'top' else (t, o + b)
        if top.startswith(bot):
            return ('top', top[len(bot):])
        if bot.startswith(top):
            return ('bot', bot[len(top):])
        return None
    for i in range(len(dominoes)):
        s = step(('top', ''), i)
        if s is None:
            continue
        if s[1] == '':
            return [i + 1]
        if s not in seen:
            seen[s] = [i + 1]
            queue.append(s)
    while queue and len(seen) < max_states:
        s = queue.popleft()
        for i in range(len(dominoes)):
            s2 = step(s, i)
            if s2 is None:
                continue
            if s2[1] == '':
                return seen[s] + [i + 1]
            if s2 not in seen:
                seen[s2] = seen[s] + [i + 1]
                queue.append(s2)
    return None

def rows(dominoes, match):
    return ''.join(dominoes[i - 1][0] for i in match), ''.join(dominoes[i - 1][1] for i in match)

fig41 = [('b', 'ca'), ('a', 'ab'), ('ca', 'a'), ('abc', 'c')]
print("Figure 4.1:", rows(fig41, [2, 1, 3, 2, 4]), " shortest:", pcp_search(fig41))
ex42 = [('a', 'baa'), ('ab', 'aa'), ('bba', 'bb')]
print("Exercise 4.2: shortest match", pcp_search(ex42), rows(ex42, [3, 2, 3, 1]))
short = [m for k in range(1, 5) for m in itertools.product(range(1, 4), repeat=k) if rows(ex42, m)[0] == rows(ex42, m)[1]]
print("all matches of length <= 4:", short)
hard = [('1000', '0'), ('01', '0'), ('1', '101'), ('00', '001')]
t0 = time.time()
m = pcp_search(hard)
print(f"hard instance: shortest match has {len(m)} dominoes, rows of length {len(rows(hard, m)[0])} ({time.time() - t0:.2f} s)")
print("it starts", m[:20], "...")

Breadth-first search over overhangs finds a *shortest* match, because the overhang is all that matters for the future. The innocent-looking four dominoes need a match of $206$ dominoes. The search space is infinite in general (overhangs can grow forever), so the search proves nothing when it fails: by Theorem 4.6 no bound on the length of the shortest match can be computed from the instance.

### P4.2

In [ ]:
B = '_'
# The palindrome machine of Chapter 3 (two-way tape): delta[q, symbol] = (r, written, move).
PAL = {('s', B): ('acc', B, 'R'), ('back', B): ('s', B, 'R')}
for c in 'ab':
    o = 'ab'.replace(c, '')
    PAL['s', c] = ('r' + c, B, 'R')
    for d in 'ab':
        PAL['r' + c, d] = ('r' + c, d, 'R')
    PAL['r' + c, B] = ('t' + c, B, 'L')
    PAL['t' + c, c] = ('back', B, 'L')
    PAL['t' + c, o] = ('rej', o, 'L')
    PAL['t' + c, B] = ('acc', B, 'L')
    PAL['back', c] = ('back', c, 'L')

def semi_thue(delta, gamma, acc='acc'):
    # Rules of Theorem 4.2. Letters: tape symbols, ('Q', state), '[', ']', 'p', 'H'. Words are tuples.
    Q = lambda q: ('Q', q)
    rules = []
    for (q, a), (r, b, D) in delta.items():
        if D == 'R':
            rules.append(((Q(q), a), (b, Q(r))))
            if a == B:
                rules.append(((Q(q), ']'), (b, Q(r), ']')))
        else:
            for c in gamma:
                rules.append(((c, Q(q), a), (Q(r), c, b)))
            rules.append((('[', Q(q), a), ('[', Q(r), B, b)))
            if a == B:
                for c in gamma:
                    rules.append(((c, Q(q), ']'), (Q(r), c, b, ']')))
                rules.append((('[', Q(q), ']'), ('[', Q(r), B, b, ']')))
    for a in gamma:
        rules.append(((Q(acc), a), (Q(acc),)))
        rules.append(((a, 'p'), ('p',)))
    rules.append(((Q(acc), ']'), ('p', ']')))
    rules.append((('[', 'p', ']'), ('H',)))
    return rules

def successors(word, rules):
    out = []
    for u, v in rules:
        for i in range(len(word) - len(u) + 1):
            if word[i:i + len(u)] == u:
                out.append(word[:i] + v + word[i + len(u):])
    return out

R = semi_thue(PAL, [B, 'a', 'b'])
print(len(R), "rules")
ok, max_succ, longest = True, 0, 0
for w in itertools.chain.from_iterable(itertools.product('ab', repeat=k) for k in range(9)):
    word, steps = ('[', ('Q', 's')) + tuple(w) + (']',), 0
    while True:
        succ = successors(word, R)
        max_succ = max(max_succ, len(succ))
        if not succ:
            break
        word, steps = succ[0], steps + 1
    ok &= (word == ('H',)) == (w == w[::-1])
    longest = max(longest, steps)
print("[q0 w] =>* H exactly for the palindromes (all words of length <= 8):", ok)
print("largest number of successors met:", max_succ, "; longest derivation:", longest, "steps")

Each derivation is a single path (never more than one successor), it follows the machine step by step, and then the clean-up rules erase the word letter by letter down to $H$. The derivations are a little longer than the machine's runs because of the clean-up.

### P4.3

In [ ]:
def derivation(word, rules):
    path = [word]
    while True:
        succ = successors(path[-1], rules)
        if not succ:
            return path
        path.append(succ[0])

def rewrite_step(x, y, rules):
    # The (unique) rule and position turning x into y.
    for u, v in rules:
        for i in range(len(x) - len(u) + 1):
            if x[i:i + len(u)] == u and x[:i] + v + x[i + len(u):] == y:
                return i, u, v

def mpcp_match(x, rules):
    # Dominoes of Step 1 as pairs of tuples; returns the list of dominoes laid in the match.
    path = derivation(x, rules)
    assert path[-1] == ('H',)
    laid = [(('#',), ('#',) + x + ('#',))]
    for xj, xk in zip(path, path[1:]):
        i, u, v = rewrite_step(xj, xk, rules)
        laid += [((c,), (c,)) for c in xj[:i]] + [(u, v)] + [((c,), (c,)) for c in xj[i + len(u):]]
        laid.append((('#',), ('#',)))
    laid.append((('H', '#', '#'), ('#',)))
    return laid

x = ('[', ('Q', 's')) + tuple('aba') + (']',)
laid = mpcp_match(x, R)
top = sum((t for t, b in laid), ())
bot = sum((b for t, b in laid), ())
print("MPCP match for aba:", len(laid), "dominoes, rows of length", len(top), "; rows equal:", top == bot)

def star(laid):
    # Step 2: star every letter; first domino *t*b*, others *t / b*, and the closing (*<>, <>).
    S, E = '*', '<>'
    st = lambda u: sum(((S, c) for c in u), ())
    ts = lambda u: sum(((c, S) for c in u), ())
    out = [(st(laid[0][0]), (S,) + ts(laid[0][1]))] + [(st(t), ts(b)) for t, b in laid[1:]]
    return out + [((S, E), (E,))]

starred = star(laid)
print("PCP match after Step 2: rows equal:",
      sum((t for t, b in starred), ()) == sum((b for t, b in starred), ()))

The match encodes the whole accepting computation on its two rows, the bottom always one configuration ahead of the top. For a rejected word (try `abb`) the derivation never reaches $H$ and there is no match at all: this is the instance an algorithm for PCP would have to recognise as hopeless.

### P4.4

In [ ]:
def tile_set(delta, gamma, q0, blank):
    # Tiles (bottom, top, left, right) of Theorem 4.11; 'e' is the empty side colour.
    states = {q for q, _ in delta} | {r for r, _, _ in delta.values()}
    T = [('beta', (q0, blank), "s'", 's'), ('beta', blank, 's', 's'), ('beta', blank, "s'", "s'"), ('beta', 'beta', 'e', 'e')]
    T += [(a, a, 'e', 'e') for a in gamma]
    for (q, a), (r, b, D) in delta.items():
        T.append(((q, a), b, 'e', (r, '>')) if D == 'R' else ((q, a), b, ('<', r), 'e'))
    for r in states:
        for c in gamma:
            T += [(c, (r, c), (r, '>'), 'e'), (c, (r, c), 'e', ('<', r))]
    return T

def tile_square(T, t0, n):
    # Backtracking on [-n, n]^2 with t0 at the origin: row 0, then upwards, then downwards.
    order = [(x, 0) for x in range(-n, n + 1)]
    order += [(x, y) for y in range(1, n + 1) for x in range(-n, n + 1)]
    order += [(x, y) for y in range(-1, -n - 1, -1) for x in range(-n, n + 1)]
    tau = {}
    def fits(t, x, y):
        b, tp, l, r = t
        nb = tau.get((x, y - 1)); nt = tau.get((x, y + 1)); nl = tau.get((x - 1, y)); nr = tau.get((x + 1, y))
        return ((nb is None or nb[1] == b) and (nt is None or nt[0] == tp) and
                (nl is None or nl[3] == l) and (nr is None or nr[2] == r))
    def go(k):
        if k == len(order):
            return True
        x, y = order[k]
        for t in ([t0] if (x, y) == (0, 0) else T):
            if fits(t, x, y):
                tau[x, y] = t
                if go(k + 1):
                    return True
                del tau[x, y]
        return False
    return tau if go(0) else None

FIG = {('A', 0): ('B', 1, 'R'), ('B', 0): ('A', 1, 'L'), ('A', 1): ('A', 1, 'L'), ('B', 1): ('A', 1, 'R')}
BB2 = {('A', 0): ('B', 1, 'R'), ('A', 1): ('B', 1, 'L'), ('B', 0): ('A', 1, 'L'), ('B', 1): ('H', 1, 'R')}
for name, delta in [("machine of Figure 4.2", FIG), ("two-state busy beaver", BB2)]:
    T = tile_set(delta, [0, 1], 'A', 0)
    t1 = time.time()
    res = [(n, tile_square(T, T[0], n) is not None) for n in range(1, 9)]
    print(name, f"({len(T)} tiles): square [-n,n]^2 tileable?", res, f"({time.time() - t1:.1f} s)")

For the machine of Figure 4.2, which never halts, every square can be tiled. The busy beaver halts in state $H$ after $6$ steps; there is no tile with bottom colour $(H,a)$, so row $7$ cannot be laid, and the squares $[-n,n]^2$ stop being tileable at $n=7$. Proposition 4.10 turns this into a semi-decision procedure for non-tileability; Theorem 4.11 says no procedure can also certify tileability in general.

### P4.5

In [ ]:
PRIMEGAME = [Fraction(*f) for f in [(17, 91), (78, 85), (19, 51), (23, 38), (29, 33), (77, 29), (95, 23),
                                     (77, 19), (1, 17), (11, 13), (13, 11), (15, 14), (15, 2), (55, 1)]]

def run_trace(prog, n, steps):
    for k in range(1, steps + 1):
        for f in prog:
            if (n * f).denominator == 1:
                n = int(n * f)
                break
        else:
            return
        yield k, n

powers = [(n.bit_length() - 1, k) for k, n in run_trace(PRIMEGAME, 2, 60000) if n & (n - 1) == 0]
print("PRIMEGAME: (exponent, step) for the powers of 2 met:", powers)

def fractran_run(prog, n, max_steps=10**6):
    for step in range(max_steps):
        for f in prog:
            if (n * f).denominator == 1:
                n = int(n * f)
                break
        else:
            return n
    return None

ADD = [Fraction(3, 2)]
MUL = [Fraction(*f) for f in [(455, 33), (11, 13), (1, 11), (3, 7), (11, 2), (1, 3)]]
print("3/2 adds:", all(fractran_run(ADD, 2**a * 3**b) == 3**(a + b) for a in range(6) for b in range(6)))
print("Exercise 4.5 multiplies:", all(fractran_run(MUL, 2**a * 3**b) == 5**(a * b) for a in range(6) for b in range(6)))

In [ ]:
def primes():
    n = 2
    while True:
        if all(n % p for p in range(2, int(n**0.5) + 1)):
            yield n
        n += 1

def compile_counter_machine(prog, k):
    # prog: dict s -> ('INC', i, j) | ('DEC', i, j, l) | ('HALT',), counters 1..k, no self-jumps.
    ps = primes()
    P = {i: next(ps) for i in range(1, k + 1)}
    Qs = {s: next(ps) for s in prog}
    out = []
    for s, ins in prog.items():
        if ins[0] == 'INC':
            out.append(Fraction(Qs[ins[2]] * P[ins[1]], Qs[s]))
        elif ins[0] == 'DEC':
            out += [Fraction(Qs[ins[2]], Qs[s] * P[ins[1]]), Fraction(Qs[ins[3]], Qs[s])]
    return out, P, Qs

# c3 = c1 * c2, with c4 as scratch: for each unit of c1, move c2 into c3 and c4, then c4 back into c2.
CM = {1: ('DEC', 1, 2, 9),
      2: ('DEC', 2, 3, 5), 3: ('INC', 3, 4), 4: ('INC', 4, 2),
      5: ('DEC', 4, 6, 1), 6: ('INC', 2, 5),
      9: ('HALT',)}
prog, P, Qs = compile_counter_machine(CM, 4)
print("compiled program:", [str(f) for f in prog])
ok = True
for a in range(5):
    for b in range(5):
        n = fractran_run(prog, Qs[1] * P[1]**a * P[2]**b)
        ok &= n == Qs[9] * P[2]**b * P[3]**(a * b)
print("the compiled program computes c3 = c1 * c2 for c1, c2 < 5:", ok)

PRIMEGAME meets $2^2$ after $19$ steps, $2^3$ after $69$, $2^5$ after $280$, and then ever more slowly: it is a terribly slow but correct prime sieve, built from trial division by counter-machine loops. The compiler of Theorem 4.16 produces fractions whose numerators and denominators are products of the instruction primes and the counter primes; the program halts on $q_9\cdot3^{b}5^{ab}$, keeping $c_2$ and the result.

### P4.6

In [ ]:
def collatz_type(prog):
    # Corollary 4.17: g(n) = a_r n with r = n mod m.
    m = math.lcm(*(f.denominator for f in prog))
    a = []
    for r in range(m):
        a.append(next((f for f in prog if r % f.denominator == 0), Fraction(1)))
    return m, a

m, a = collatz_type(MUL)
print("multiplication program as a Collatz-type map: modulus", m)
n = 2**3 * 3**4
while True:
    n2 = int(a[n % m] * n)
    if n2 == n:
        break
    n = n2
print("fixed point reached from 2^3 3^4:", n, "= 5^12:", n == 5**12)

def stopping_time(n):
    k = 0
    while n != 1:
        n = n // 2 if n % 2 == 0 else 3 * n + 1
        k += 1
    return k
records, best = [], -1
for n in range(1, 10**5):
    s = stopping_time(n)
    if s > best:
        best = s
        records.append((n, s))
print("Collatz record stopping times below 10^5:", records[-8:])

In [ ]:
a_, b_, low, hist = 8, 0, 0, []
t0 = time.time()
for k in range(10**5):
    if a_ % 2 == 0:
        b_ += 2
    else:
        b_ -= 1
    if b_ == -1:
        print("Antihydra halts at iteration", k)
        break
    a_ += a_ // 2
    if k % 1000 == 0:
        hist.append((k, b_))
print(f"after 10^5 iterations: b = {b_}, a has {a_.bit_length()} bits ({time.time() - t0:.1f} s)")
print("smallest b after the start:", min(b for k, b in hist[1:]))
plt.figure(figsize=(6, 3.2))
plt.plot([k for k, _ in hist], [b for _, b in hist])
plt.plot([0, 10**5], [0, 0.5 * 10**5], '--', label='drift $k/2$ (fair coin)')
plt.xlabel('iteration'); plt.ylabel('b'); plt.legend(); plt.title('Antihydra: the counter b'); plt.show()

The multiplication program, seen as a map $n\mapsto a_{n\bmod m}\,n$ with $m=3\cdot7\cdot11\cdot13\cdot2=6006$, reaches its fixed point $5^{12}$. Collatz records grow slowly ($77031$ needs $350$ steps). Antihydra's counter $b$ follows the drift of a fair coin, $+2$ or $-1$ with probability $\frac12$, about $\frac12$ per step, and is far from $-1$ after $10^5$ iterations; if the parities of $a$ behave randomly forever, the probability of ever reaching $-1$ from this height is astronomically small, but no proof is known that it never happens.

### P4.7

In [ ]:
def pell(a, n):
    d = a * a - 1
    x, y = 1, 0
    out = [(x, y)]
    for _ in range(n):
        x, y = a * x + d * y, x + a * y
        out.append((x, y))
    return out

for a in range(2, 7):
    d = a * a - 1
    sols = [(x, y) for y in range(0, 3000) for x in [math.isqrt(d * y * y + 1)] if x * x - d * y * y == 1]
    assert sols == [p for p in pell(a, 30) if p[1] < 3000], a
print("Lemma 4.23: the solutions with y < 3000 are exactly the (x_n, y_n), for a = 2..6")

ok_a = ok_b = ok_div = True
for a in range(2, 12):
    P = pell(a, 25)
    for n, (x, y) in enumerate(P):
        ok_a &= (y - n) % (a - 1) == 0
        for k in range(0, 8):
            N = 2 * a * k - k * k - 1
            if N != 0:
                ok_b &= (x - (a - k) * y - k**n) % N == 0
        if n >= 1:
            ok_div &= all(P[n * j][1] % y == 0 for j in range(1, 25 // n + 1))
print("Lemma 4.24(a):", ok_a, " (b):", ok_b, " Exercise 4.8 (y_n | y_nk):", ok_div)

def power_via_pell(k, n):
    # Proposition 4.26: a = y_{n+1}(k+1) + k + 1, then k^n is the residue of x_n(a) - (a-k) y_n(a).
    a = pell(k + 1, n + 1)[n + 1][1] + k + 1
    x, y = pell(a, n)[n]
    N = 2 * a * k - k * k - 1
    return (x - (a - k) * y) % N
print("k^n recovered from Pell sequences for k = 1..6, n = 0..8:",
      all(power_via_pell(k, n) == k**n for k in range(1, 7) for n in range(9)))

Everything checks. Note how wasteful the recovery of $k^n$ is: to compute $6^8$ it builds Pell numbers $x_8(a)$ for $a$ of more than seven digits. Proposition 4.26 is not an algorithm for powers; it is a proof that the graph of exponentiation can be described by a polynomial equation with existential unknowns, which is what Hilbert's tenth problem needed.

### P4.8

In [ ]:
def semi_decide(P, nvars, bound):
    # Enumerate N^nvars by increasing sum of coordinates (Proposition 4.19(c)), up to a bound on the sum.
    for s in range(bound + 1):
        for c in itertools.combinations(range(s + nvars - 1), nvars - 1):
            v = [b - a - 1 for a, b in zip((-1,) + c, c + (s + nvars - 1,))]
            if P(*v) == 0:
                return v, s
    return None

for d in [2, 3, 7, 13, 61]:
    t0 = time.time()
    r = semi_decide(lambda x, y: x * x - d * y * y - 1 if y > 0 else 1, 2, 3000)
    print(f"d = {d}: first solution with y > 0 found by enumeration: {r} ({time.time() - t0:.2f} s)")
print("continued fractions (Tome 1, Chapter 8) give for d = 61: x = 1766319049, y = 226153980")

In [ ]:
# Corollary 4.22 applied to the composites: x composite <=> exists y, z: x - (y+2)(z+2) = 0.
Pc = lambda x, y, z: x - (y + 2) * (z + 2)
Qc = lambda x, y, z: (x + 1) * (1 - Pc(x, y, z)**2) - 1
vals = sorted({Qc(x, y, z) for x in range(60) for y in range(30) for z in range(30)})
print("positive values of Q below 60:", [v for v in vals if 0 < v < 60])
print("negative values (a few):", vals[:3], "...")

The enumeration finds the small solutions at once, but for $d=61$ the smallest solution has $y=226153980$, and the enumeration gives up within its bound: a semi-decision procedure never knows whether to keep waiting. The polynomial $Q$ takes as positive values exactly the composite numbers, and wild negative values elsewhere; the polynomial of Jones, Sato, Wada and Wiens does the same for the primes, with $26$ variables.